# Impact of Stride — Section 6.3

Change the stride **S** to see which positions the filter visits and how that reduces the output size.

**Formula:** Output = ⌊(N − F) / S⌋ + 1  (with no padding)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets


def draw_stride(N=7, F=3, S=1):
    raw = (N - F) / S + 1
    valid = (raw == int(raw)) and raw > 0
    O = int(raw) if valid else None

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --- Input grid showing anchor positions ---
    ax = axes[0]
    max_anchor = N - F

    for i in range(N):
        for j in range(N):
            # Is this a valid anchor with current stride?
            is_anchor = valid and (i <= max_anchor and j <= max_anchor
                                   and i % S == 0 and j % S == 0)
            # Would be visited with S=1 but skipped?
            would_visit = (i <= max_anchor and j <= max_anchor)
            is_skipped = S > 1 and would_visit and not is_anchor

            if is_anchor:
                color = '#bfdbfe'
            elif is_skipped:
                color = '#fee2e2'
            else:
                color = '#e2e8f0'

            rect = patches.FancyBboxPatch((j, N - 1 - i), 1, 1,
                                          boxstyle='square,pad=0', fc=color, ec='white', lw=1)
            ax.add_patch(rect)

            if is_anchor:
                ax.text(j + 0.5, N - 0.5 - i, '●', ha='center', va='center',
                        fontsize=9, color='#1d4ed8')

    ax.set_xlim(0, N); ax.set_ylim(0, N)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect('equal')

    legend_items = [
        patches.Patch(fc='#bfdbfe', label='Visited anchor'),
        patches.Patch(fc='#e2e8f0', label='Input cell'),
    ]
    if S > 1:
        legend_items.append(patches.Patch(fc='#fee2e2', label='Skipped (stride)'))
    ax.legend(handles=legend_items, loc='upper right', fontsize=9)

    if valid:
        title = f'Input {N}×{N}  |  Filter {F}×{F}  |  Stride S={S}\n{O*O} anchors visited (●)'
    else:
        title = f'Input {N}×{N}  |  Filter {F}×{F}  |  Stride S={S}  →  Invalid!'
    ax.set_title(title, fontsize=11)

    # --- Output grid ---
    ax = axes[1]
    if valid:
        for i in range(O):
            for j in range(O):
                rect = patches.FancyBboxPatch((j, O - 1 - i), 1, 1,
                                              boxstyle='square,pad=0', fc='#a7f3d0', ec='white', lw=1)
                ax.add_patch(rect)
        ax.set_xlim(0, O); ax.set_ylim(0, O)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_aspect('equal')
        ax.set_title(f'Output: {O}×{O}  ({O*O} values)', fontsize=11)

        s1O = N - F + 1
        reduction = (1 - O * O / (s1O * s1O)) * 100
        fig.suptitle(
            f'⌊({N}−{F})/{S}⌋+1 = {O}  |  vs S=1 output {s1O}×{s1O}: {reduction:.0f}% fewer values',
            fontsize=11, color='#059669', y=0.02
        )
    else:
        ax.text(0.5, 0.5, f'Invalid stride S={S}\n(N−F)={N-F} not divisible by S',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=12, color='#dc2626')
        ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    plt.show()


widgets.interact(
    draw_stride,
    N=widgets.IntSlider(min=5, max=12, step=1, value=7, description='Input N', continuous_update=False),
    F=widgets.IntSlider(min=1, max=5, step=1, value=3, description='Filter F', continuous_update=False),
    S=widgets.IntSlider(min=1, max=5, step=1, value=1, description='Stride S', continuous_update=False),
)